# 04 — GBDT Benchmark (Adim 6)

XGBoost / LightGBM / CatBoost — profit (Model 1) ve route (Model 2) icin.

**Cikti:**
- `models/model_profit.pkl`
- `models/model_route.pkl`
- `data/processed/gbm_results.json`

In [ ]:
from __future__ import annotations
import sys, json, warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
print('repo root:', REPO_ROOT)

SEED = 42

In [ ]:
from ml.src.train_profit import train_profit_models
from ml.src.train_route import train_route_models

df = pd.read_parquet(REPO_ROOT / 'data/processed/training_set_v1.parquet')
print('df shape:', df.shape)

## Profit (Model 1)

In [ ]:
metrics_p, best_p, models_p = train_profit_models(df, seed=SEED)
for k, v in metrics_p.items():
    print(f'{k}: rmse={v["rmse"]:.0f}  mae={v["mae"]:.0f}  r2={v["r2"]:.4f}  mape={v["mape"]:.3f}  train_sec={v["train_sec"]:.1f}')
print('BEST profit:', best_p)

joblib.dump(models_p[best_p], REPO_ROOT / 'models/model_profit.pkl')
print('saved models/model_profit.pkl')

## Route (Model 2)

In [ ]:
metrics_r, best_r, models_r, classes = train_route_models(df, seed=SEED)
for k, v in metrics_r.items():
    print(f'{k}: acc={v["accuracy"]:.4f}  macro_f1={v["macro_f1"]:.4f}  top2={v.get("top2_accuracy",0):.4f}  train_sec={v["train_sec"]:.1f}')
print('BEST route:', best_r)
print('classes:', classes)

joblib.dump(models_r[best_r], REPO_ROOT / 'models/model_route.pkl')
print('saved models/model_route.pkl')

In [ ]:
# Save gbm_results.json
results = {
    'profit': {**metrics_p, 'best_model': best_p},
    'route':  {**metrics_r, 'best_model': best_r, 'classes': classes},
    'seed': SEED,
}
Path(REPO_ROOT / 'data/processed').mkdir(parents=True, exist_ok=True)
out_path = REPO_ROOT / 'data/processed/gbm_results.json'
out_path.write_text(json.dumps(results, indent=2, default=float))
print('saved:', out_path)
results

## Feature importance — Top 15 (best profit model)

In [ ]:
# CatBoost'ta feature_importances_ ve feature_names_ vardir; XGB/LGB icin de mevcut
best_model = models_p[best_p]
if hasattr(best_model, 'get_feature_importance'):
    fi = best_model.get_feature_importance()
    names = best_model.feature_names_
elif hasattr(best_model, 'feature_importances_'):
    fi = best_model.feature_importances_
    names = [f'f{i}' for i in range(len(fi))]
    if hasattr(best_model, 'feature_names_in_'):
        names = list(best_model.feature_names_in_)
else:
    fi, names = None, None

if fi is not None:
    order = np.argsort(fi)[::-1][:15]
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh([names[i] for i in order][::-1], [fi[i] for i in order][::-1])
    ax.set_title(f'profit ({best_p}) — Top-15 feature importance')
    plt.tight_layout()
    plt.show()

## Predicted vs actual scatter (profit)

In [ ]:
# Best profit model'i kullanarak test set tahmin
from ml.src.train_profit import _unique_feature_cols
from ml.src.baselines import stratified_split
from ml.src.feature_pipeline import build_feature_pipeline

feat_cols = _unique_feature_cols(df)
_, _, test_df = stratified_split(df, 'expected_profit_try', stratify_cols=['raw_material','processing_route_candidate'], seed=SEED)

if best_p == 'catboost':
    df_test_cb = test_df[feat_cols].copy()
    from ml.src.feature_pipeline import HIGH_CARDINALITY, LOW_CARDINALITY
    cat_features = [c for c in (LOW_CARDINALITY + HIGH_CARDINALITY) if c in feat_cols]
    for c in cat_features:
        df_test_cb[c] = df_test_cb[c].fillna('MISSING').astype(str)
    for c in df_test_cb.columns:
        if c in cat_features: continue
        if df_test_cb[c].dtype.kind in 'fi':
            df_test_cb[c] = df_test_cb[c].fillna(df_test_cb[c].median())
        if df_test_cb[c].dtype == bool:
            df_test_cb[c] = df_test_cb[c].astype(int)
    y_pred = best_model.predict(df_test_cb)
else:
    pipe = build_feature_pipeline('xgboost', df_columns=feat_cols)
    pipe.fit(df[feat_cols])
    y_pred = best_model.predict(pipe.transform(test_df[feat_cols]))

y_true = test_df['expected_profit_try'].values
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_true, y_pred, alpha=0.4, s=10)
lo, hi = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
ax.plot([lo, hi], [lo, hi], 'r--', label='y = x')
ax.set_xlabel('actual profit (TRY)')
ax.set_ylabel('predicted profit (TRY)')
ax.set_title(f'{best_p}: predicted vs actual')
ax.legend()
plt.tight_layout()
plt.show()

## Confusion matrix (best route)

In [ ]:
from sklearn.metrics import confusion_matrix
from ml.src.train_route import _unique_feature_cols as _route_cols

feat_cols_r = _route_cols(df)
_, _, test_r = stratified_split(df, 'recommended_route_label', stratify_cols=['raw_material'], seed=SEED)
y_test_r = test_r['recommended_route_label'].values

best_r_model = models_r[best_r]
if best_r == 'catboost':
    df_test_r = test_r[feat_cols_r].copy()
    from ml.src.feature_pipeline import HIGH_CARDINALITY, LOW_CARDINALITY
    cat_features = [c for c in (LOW_CARDINALITY + HIGH_CARDINALITY) if c in feat_cols_r]
    for c in cat_features:
        df_test_r[c] = df_test_r[c].fillna('MISSING').astype(str)
    for c in df_test_r.columns:
        if c in cat_features: continue
        if df_test_r[c].dtype.kind in 'fi':
            df_test_r[c] = df_test_r[c].fillna(df_test_r[c].median())
        if df_test_r[c].dtype == bool:
            df_test_r[c] = df_test_r[c].astype(int)
    y_pred_r = best_r_model.predict(df_test_r).flatten()
else:
    pipe_r = build_feature_pipeline('xgboost', df_columns=feat_cols_r)
    pipe_r.fit(df[feat_cols_r])
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder().fit(classes)
    y_pred_r = le.inverse_transform(best_r_model.predict(pipe_r.transform(test_r[feat_cols_r])))

cm = confusion_matrix(y_test_r, y_pred_r, labels=classes)
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(classes))); ax.set_yticks(range(len(classes)))
ax.set_xticklabels(classes, rotation=75, ha='right', fontsize=8)
ax.set_yticklabels(classes, fontsize=8)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title(f'{best_r} — confusion matrix')
plt.colorbar(im)
plt.tight_layout()
plt.show()